# ACE vs EXFOR: Angular Distribution Comparison

This notebook demonstrates how to combine angular distribution data from two sources:
- **ACE files**: Evaluated nuclear data (theoretical/evaluated curves)
- **EXFOR database**: Experimental measurements with uncertainties

We compare Fe-56 elastic scattering (n,el) at 1.3 MeV.

## Key Considerations

**Unit Consistency**: Both ACE and EXFOR data use the same unit: $d\sigma/d\Omega$ (barns/steradian).
- For ACE data, use `cross_section_unit='b/sr'` in `to_plot_data()` to get the correct unit
- This handles the internal conversion from $d\sigma/d\mu$ to $d\sigma/d\Omega = \frac{1}{2\pi} \cdot d\sigma/d\mu$

**Frame Alignment**: Both datasets are converted to the center-of-mass (CM) frame.

**Plotting**: Using `PlotBuilder.add_data()` for both datasets ensures proper axis limits.

## 1. Setup and Imports

In [ ]:
import kika
import kika.exfor as exfor
from kika.exfor import X4ProDatabase, read_all_exfor
from kika.plotting import PlotBuilder
import matplotlib.pyplot as plt
import numpy as np

# Configure EXFOR database path
exfor.configure(db_path=r"C:\Users\Usuario\BaradDur\EXFOR\x4sqlite-20251105-c\x4sqlite1.db")

## 2. Define Target Energy and Tolerance

In [ ]:
# Target energy for comparison
TARGET_ENERGY = 1.3  # MeV
ENERGY_TOLERANCE = 0.02  # MeV (20 keV tolerance for EXFOR matching)

print(f"Target energy: {TARGET_ENERGY} MeV")
print(f"Energy tolerance: {ENERGY_TOLERANCE*1000:.0f} keV")

## 3. Load ACE Data for Fe-56

In [ ]:
# Load Fe-56 ACE file
ace_file = r'C:\Users\Usuario\BaradDur\Dev\kika\files\ace\260560_80.02c'
ace = kika.read_ace(ace_file)

print(f"Loaded ACE file for ZAID: {ace.header.zaid}")
print(f"Temperature: {ace.header.temperature:.4e} MeV")

In [ ]:
# Get angular distribution plot data from ACE at target energy
# Using normalize_to_xs=True and cross_section_unit='b/sr' to get dsigma/dOmega (b/sr)
# This makes it directly comparable with EXFOR data
ace_plot_data = ace.angular_distributions.to_plot_data(
    mt=2,  # Elastic scattering
    energy=TARGET_ENERGY,
    ace=ace,
    interpolate=True,
    num_points=200,
    normalize_to_xs=True,
    cross_section_unit='b/sr',  # Same unit as EXFOR
    label='ENDF/B-VIII.0 (ACE)',
    color='black',
    linewidth=2
)

print(f"ACE PlotData created:")
print(f"  x (cosine) range: [{ace_plot_data.x.min():.3f}, {ace_plot_data.x.max():.3f}]")
print(f"  y (dsigma/dOmega) range: [{ace_plot_data.y.min():.4f}, {ace_plot_data.y.max():.4f}] b/sr")

## 4. Load EXFOR Experimental Data

In [ ]:
# Load all Fe-56 elastic scattering experiments
print("Loading Fe-56 elastic scattering data from EXFOR...")
fe56_experiments = read_all_exfor(target="Fe56", mt=2, group_by_energy=False)

# Also load natural Fe (Fe-56 is 91.75% of natural Fe)
print("Loading natural Fe elastic scattering data from EXFOR...")
fe_nat_experiments = read_all_exfor(target="Fe", mt=2, group_by_energy=False)

print(f"\nFe-56: {len(fe56_experiments)} unique experiments")
print(f"Fe-nat: {len(fe_nat_experiments)} unique experiments")

In [ ]:
def find_experiments_near_energy(experiments_dict, target_e, tolerance):
    """
    Find experiments that have data near the target energy.
    
    Parameters:
        experiments_dict: Dict from read_all_exfor(group_by_energy=False)
        target_e: Target energy in MeV
        tolerance: Absolute tolerance in MeV
    
    Returns:
        List of ExforAngularDistribution objects with data near target_e
    """
    matching = []
    for exp_id, exp in experiments_dict.items():
        energies = exp.energies(unit="MeV")
        if np.any(np.abs(energies - target_e) <= tolerance):
            matching.append(exp)
    return matching

# Find experiments near target energy
fe56_near = find_experiments_near_energy(fe56_experiments, TARGET_ENERGY, ENERGY_TOLERANCE)
fe_nat_near = find_experiments_near_energy(fe_nat_experiments, TARGET_ENERGY, ENERGY_TOLERANCE)

print(f"Experiments with data near {TARGET_ENERGY} MeV (+/- {ENERGY_TOLERANCE*1000:.0f} keV):")
print(f"  Fe-56: {len(fe56_near)} experiments")
print(f"  Fe-nat: {len(fe_nat_near)} experiments")

# Show experiment details
print(f"\nFe-56 experiments:")
for exp in fe56_near:
    print(f"  {exp.entry}/{exp.subentry}: {exp.label}")

print(f"\nFe-nat experiments:")
for exp in fe_nat_near:
    nat_tag = "[nat]" if exp.is_natural_target else ""
    print(f"  {exp.entry}/{exp.subentry}: {exp.label} {nat_tag}")

## 5. Create Combined Plot: ACE + EXFOR

In [ ]:
# Create figure and use PlotBuilder for all data
# This ensures axis limits are calculated correctly for both ACE and EXFOR
builder = PlotBuilder(figsize=(10, 7))

# Add ACE evaluated curve using PlotBuilder (not ax.plot)
builder.add_data(ace_plot_data)

# Plot EXFOR experimental data with error bars
markers = ['o', 's', '^', 'v', 'D', 'p', 'h', '*']
colors = plt.cm.tab10.colors

all_experiments = fe56_near + fe_nat_near
plotted_count = 0

for exp in all_experiments:
    plot_data = exp.to_plot_data(
        energy=TARGET_ENERGY,
        tolerance=ENERGY_TOLERANCE,
        frame='CM',  # Convert to CM frame
        angle_unit='cos',
        cross_section_unit='b/sr',
        uncertainty=True,
        include_natural_tag=True,
        marker=markers[plotted_count % len(markers)],
        color=colors[plotted_count % len(colors)],
        markersize=6,
        capsize=3,
        alpha=0.8,
    )
    
    if plot_data is not None:
        builder.add_data(plot_data)
        plotted_count += 1

# Configure and build plot
fig = (builder
       .set_labels(
           title=f"Fe-56(n,el) Angular Distribution @ E = {TARGET_ENERGY} MeV\nACE Evaluation vs EXFOR Experiments (CM Frame)",
           x_label=r"$\cos(\theta_{CM})$",
           y_label=r"$d\sigma/d\Omega$ (b/sr)"
       )
       .set_scales(log_y=True)
       .set_legend(loc='upper left')
       .build())

plt.tight_layout()
plt.show()

print(f"\nPlotted {plotted_count} EXFOR experiments along with ACE evaluation")

## 6. Alternative View: Linear Scale

In [ ]:
# Same plot with linear y-scale for different perspective
builder = PlotBuilder(figsize=(10, 7))

# Add ACE curve
builder.add_data(ace_plot_data)

# EXFOR data
plotted_count = 0
for exp in all_experiments:
    plot_data = exp.to_plot_data(
        energy=TARGET_ENERGY,
        tolerance=ENERGY_TOLERANCE,
        frame='CM',
        angle_unit='cos',
        cross_section_unit='b/sr',
        uncertainty=True,
        include_natural_tag=True,
        marker=markers[plotted_count % len(markers)],
        color=colors[plotted_count % len(colors)],
        markersize=6,
        capsize=3,
        alpha=0.8,
    )
    
    if plot_data is not None:
        builder.add_data(plot_data)
        plotted_count += 1

fig = (builder
       .set_labels(
           title=f"Fe-56(n,el) Angular Distribution @ E = {TARGET_ENERGY} MeV\nLinear Scale (CM Frame)",
           x_label=r"$\cos(\theta_{CM})$",
           y_label=r"$d\sigma/d\Omega$ (b/sr)"
       )
       .set_scales(log_y=False)
       .set_legend(loc='upper left')
       .build())

plt.tight_layout()
plt.show()

## 7. Verify Unit Consistency

Let's verify our unit conversion by checking that the integrated cross-section makes sense.

In [ ]:
# Integrate ACE differential cross-section to get total elastic XS
# sigma_total = integral of (dsigma/dOmega) dOmega = 2*pi * integral of (dsigma/dOmega) dmu
ace_cosine = ace_plot_data.x
ace_dsigma_dOmega = ace_plot_data.y
integrated_xs = 2 * np.pi * np.trapezoid(ace_dsigma_dOmega, ace_cosine)

# Get total elastic cross-section from ACE at target energy
# Use the reaction dictionary to access MT=2 (elastic) cross-section
elastic_reaction = ace.cross_section.reaction[2]
xs_df = elastic_reaction.to_dataframe()
energies = xs_df['Energy'].values
xs_values = xs_df['Cross Section'].values

# Find closest energy
idx = np.argmin(np.abs(energies - TARGET_ENERGY))
total_xs_ace = xs_values[idx]

print(f"Integration of angular distribution: {integrated_xs:.4f} barns")
print(f"Total elastic XS from ACE at {energies[idx]:.4f} MeV: {total_xs_ace:.4f} barns")
print(f"Ratio: {integrated_xs/total_xs_ace:.4f} (should be close to 1.0)")

## 8. Energy Folding for TOF Comparison

In Time-of-Flight (TOF) experiments, neutron energy is determined by measuring flight time. This introduces energy uncertainty:

**Energy Resolution Formula:**
$$\sigma_E = E \cdot 2 \cdot \frac{\delta t}{t} = E \cdot 2 \cdot \frac{\delta t \cdot v}{L}$$

Where:
- $E$ = neutron energy (MeV)
- $\delta t$ = time resolution (nanoseconds)  
- $L$ = flight path distance (meters)
- $v$ = neutron velocity

When comparing evaluated data (ACE) with TOF experiments, the evaluation should be convolved with a Gaussian kernel of width $\sigma_E$ to account for this energy spread.

**Default TOF parameters (GELINA facility):**
- Flight path: 27.037 m
- Time resolution: 5 ns

In [ ]:
# Import energy resolution function for verification
from kika.ace.classes.angular_distribution.container import _compute_energy_resolution_tof

# Calculate energy resolution at target energy with default TOF parameters
sigma_E_default = _compute_energy_resolution_tof(TARGET_ENERGY)  # Uses 27.037m, 5ns
print(f"Energy resolution at {TARGET_ENERGY} MeV (GELINA defaults):")
print(f"  σE = {sigma_E_default*1000:.2f} keV")

# Calculate for different TOF setups
sigma_E_orela = _compute_energy_resolution_tof(TARGET_ENERGY, flight_path_m=40.0, delta_t_ns=8.0)
print(f"\nEnergy resolution at {TARGET_ENERGY} MeV (ORELA-like, 40m, 8ns):")
print(f"  σE = {sigma_E_orela*1000:.2f} keV")

# Verify at benchmark energies mentioned in the plan
print("\nBenchmark verification:")
print(f"  At 1 MeV, 27.037m, 5ns: σE = {_compute_energy_resolution_tof(1.0)*1000:.1f} keV (expected ~5.4 keV)")
print(f"  At 10 MeV, 27.037m, 5ns: σE = {_compute_energy_resolution_tof(10.0)*1000:.1f} keV (expected ~170 keV)")

In [ ]:
# Compare unfolded vs energy-folded ACE angular distributions
# This demonstrates how energy folding smooths the distribution

# Original ACE curve (no folding)
ace_unfolded = ace.angular_distributions.to_plot_data(
    mt=2,
    energy=TARGET_ENERGY,
    ace=ace,
    interpolate=True,
    num_points=200,
    normalize_to_xs=True,
    cross_section_unit='b/sr',
    label='ACE (unfolded)',
    color='black',
    linewidth=2
)

# ACE with energy folding (default GELINA parameters: 27.037m, 5ns)
ace_folded = ace.angular_distributions.to_plot_data(
    mt=2,
    energy=TARGET_ENERGY,
    ace=ace,
    interpolate=True,
    num_points=200,
    normalize_to_xs=True,
    cross_section_unit='b/sr',
    energy_folding=True,  # Uses default TOF parameters
    label=f'ACE (folded, σE={sigma_E_default*1000:.1f} keV)',
    color='blue',
    linewidth=2,
    linestyle='--'
)

# ACE with custom TOF parameters (ORELA-like: 40m, 8ns)
ace_folded_orela = ace.angular_distributions.to_plot_data(
    mt=2,
    energy=TARGET_ENERGY,
    ace=ace,
    interpolate=True,
    num_points=200,
    normalize_to_xs=True,
    cross_section_unit='b/sr',
    energy_folding=True,
    tof=(40.0, 8.0),  # ORELA-like parameters
    label=f'ACE (folded, σE={sigma_E_orela*1000:.1f} keV)',
    color='red',
    linewidth=2,
    linestyle=':'
)

# Create comparison plot
builder = PlotBuilder(figsize=(10, 7))
builder.add_data(ace_unfolded)
builder.add_data(ace_folded)
builder.add_data(ace_folded_orela)

fig = (builder
       .set_labels(
           title=f"Energy Folding Effect on Angular Distribution @ E = {TARGET_ENERGY} MeV",
           x_label=r"$\cos(\theta_{CM})$",
           y_label=r"$d\sigma/d\Omega$ (b/sr)"
       )
       .set_scales(log_y=True)
       .set_legend(loc='upper left')
       .build())

plt.tight_layout()
plt.show()

print(f"Energy folding smooths the angular distribution to account for")
print(f"experimental energy resolution in TOF measurements.")

In [ ]:
# Main use case: Compare energy-folded ACE with EXFOR experimental data
# This is how you properly compare evaluated data with TOF experimental measurements

# ACE with energy folding for proper comparison with EXFOR TOF data
ace_folded_for_exfor = ace.angular_distributions.to_plot_data(
    mt=2,
    energy=TARGET_ENERGY,
    ace=ace,
    interpolate=True,
    num_points=200,
    normalize_to_xs=True,
    cross_section_unit='b/sr',
    energy_folding=True,  # Enable folding - uses default (27.037m, 5ns)
    label='ENDF/B-VIII.0 (energy folded)',
    color='black',
    linewidth=2
)

# Create comparison plot: Folded ACE vs EXFOR experiments
builder = PlotBuilder(figsize=(10, 7))

# Add the energy-folded ACE curve
builder.add_data(ace_folded_for_exfor)

# Add EXFOR experimental data
plotted_count = 0
for exp in all_experiments:
    plot_data = exp.to_plot_data(
        energy=TARGET_ENERGY,
        tolerance=ENERGY_TOLERANCE,
        frame='CM',
        angle_unit='cos',
        cross_section_unit='b/sr',
        uncertainty=True,
        include_natural_tag=True,
        marker=markers[plotted_count % len(markers)],
        color=colors[plotted_count % len(colors)],
        markersize=6,
        capsize=3,
        alpha=0.8,
    )
    
    if plot_data is not None:
        builder.add_data(plot_data)
        plotted_count += 1

fig = (builder
       .set_labels(
           title=f"Fe-56(n,el) @ E = {TARGET_ENERGY} MeV\nEnergy-Folded ACE vs EXFOR (proper TOF comparison)",
           x_label=r"$\cos(\theta_{CM})$",
           y_label=r"$d\sigma/d\Omega$ (b/sr)"
       )
       .set_scales(log_y=True)
       .set_legend(loc='upper left')
       .build())

plt.tight_layout()
plt.show()

print(f"Energy folding accounts for the experimental energy resolution in TOF measurements.")
print(f"This provides a fairer comparison between evaluated data and experiments.")

## Summary

This notebook demonstrated how to:
1. Load angular distribution data from ACE files using `ace.angular_distributions.to_plot_data()`
2. Load experimental angular distribution data from EXFOR using `exp.to_plot_data()`
3. Use `cross_section_unit='b/sr'` to ensure consistent units between ACE and EXFOR
4. Align reference frames (both in CM) using `frame='CM'` for EXFOR data
5. Use `PlotBuilder.add_data()` for all datasets to ensure proper axis limits
6. Create combined plots comparing theory vs experiment
7. Apply **energy folding** to ACE data using `energy_folding=True` to account for TOF energy resolution
   - Default parameters: GELINA facility (27.037 m, 5 ns)
   - Custom parameters via `tof=(flight_path_m, delta_t_ns)`

The comparison allows validation of nuclear data evaluations against experimental measurements, with proper treatment of experimental energy resolution.